In [0]:
 %run ../config/config

In [0]:
import numpy as np
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType,
    TimestampType, MapType, ArrayType, IntegerType,
)
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    confusion_matrix, classification_report,
)
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots


#### 1. Load all model predictions

In [0]:
all_results = {}

for model_name in model_names:
    try:
        tbl = f"{catalog}.{gold_schema}.{model_name}_predictions"
        pdf = spark.table(tbl).toPandas()
        all_results[model_name] = {
            "predicted": pdf["predicted"].values,
            "actual": pdf["actual"].values,
            "run_id": pdf["run_id"].iloc[0] if "run_id" in pdf else "",
        }
        print(f"Loaded {model_name}: {len(pdf):,} predictions")
    except Exception as e:
        print(f"Load result fail{model_name}: {type(e).__name__}: {e}")

#### 2. Compute metrics 

In [0]:
phase_names = phase_name_list
metrics_rows = []

comparison_table = []

for model_name, data in all_results.items():
    preds = data["predicted"]
    actuals = data["actual"]

    acc = accuracy_score(actuals, preds)
    macro_f1 = f1_score(actuals, preds, average="macro", zero_division=0)
    weighted_f1 = f1_score(actuals, preds, average="weighted", zero_division=0)

    # Per-phase metrics
    per_f1 = f1_score(actuals, preds, average=None, labels=range(number_of_phases), zero_division=0)
    per_prec = precision_score(actuals, preds, average=None, labels=range(number_of_phases), zero_division=0)
    per_rec = recall_score(actuals, preds, average=None, labels=range(number_of_phases), zero_division=0)

    cm = confusion_matrix(actuals, preds, labels=range(number_of_phases))

    per_phase_f1 = {phase_names[i]: float(per_f1[i]) for i in range(number_of_phases)}
    per_phase_prec = {phase_names[i]: float(per_prec[i]) for i in range(number_of_phases)}
    per_phase_rec = {phase_names[i]: float(per_rec[i]) for i in range(number_of_phases)}

    metrics_rows.append((
        model_name,
        data.get("run_id", ""),
        datetime.now(),
        "test",
        float(acc),
        float(macro_f1),
        float(weighted_f1),
        per_phase_f1,
        per_phase_prec,
        per_phase_rec,
        cm.tolist(),
    ))

    comparison_table.append({
        "Model": model_name,
        "Accuracy": acc,
        "Macro F1": macro_f1,
        "Weighted F1": weighted_f1,
        **{f"F1_{name}": float(per_f1[i]) for i, name in enumerate(phase_names)},
    })

    print(f"Model: {model_name}")
    print(f"Accuracy: {acc:.4f}  Macro F1: {macro_f1:.4f}  Weighted F1: {weighted_f1:.4f}")

#### 3. Comparison table

In [0]:
import pandas as pd

comp_df = (
    pd.DataFrame(comparison_table)
    .sort_values("Macro F1", ascending=False)
    .reset_index(drop=True)
)
print("Model Comparision")
print(comp_df.to_string(index=False, float_format="%.5f"))

#### 4. Confusion matrix heatmaps

In [0]:
for model_name, data in all_results.items():
    cm = confusion_matrix(data["actual"], data["predicted"], labels=range(number_of_phases))

    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)

    fig = go.Figure(data=go.Heatmap(
        z=cm_norm,
        x=phase_names,
        y=phase_names,
        colorscale="Blues",
        text=cm,
        texttemplate="%{text}",
        textfont={"size": 10},
        hoverongaps=False,
    ))

    fig.update_layout(
        title=f"Confusion Matrix - {model_name}",
        xaxis_title="Predicted",
        yaxis_title="Actual",
        width=700,
        height=600,
    )

    displayHTML(fig.to_html())

#### 5. Per-phase F1 comparison chart

In [0]:
fig = go.Figure()

for row in comparison_table:
    model = row["Model"]
    f1_values = [row.get(f"F1_{name}", 0) for name in phase_names]
    fig.add_trace(go.Bar(name=model, x=phase_names, y=f1_values))

fig.update_layout(
    title="Per-Phase F1 Score by Model",
    xaxis_title="Wyckoff Phase",
    yaxis_title="F1 Score",
    barmode="group",
    width=1000,
    height=500,
    legend_title="Model",
)

displayHTML(fig.to_html())

#### 6. Overall metrics bar chart

In [0]:
fig = make_subplots(rows=1, cols=3, subplot_titles=["Accuracy", "Macro F1", "Weighted F1"])

models = [r["Model"] for r in comparison_table]
colors = px.colors.qualitative.Set2[:len(models)]

for i, metric in enumerate(["Accuracy", "Macro F1", "Weighted F1"]):
    values = [r[metric] for r in comparison_table]
    fig.add_trace(
        go.Bar(x=models, y=values, marker_color=colors, showlegend=False,
               text=[f"{v:.3f}" for v in values], textposition="auto"),
        row=1, col=i+1,
    )

fig.update_layout(title="Model Comparison - Overall Metrics", width=1200, height=400)
displayHTML(fig.to_html())

In [0]:
%sql
SELECT actual, predicted, COUNT(*) AS n,
         ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY predicted), 1) AS pct_of_predicted
FROM wyckoff_catalog.gold.cnn_lstm_predictions
WHERE predicted IN (3, 4, 8, 9)  -- accum_D, accum_E, distrib_D, distrib_E
GROUP BY actual, predicted
ORDER BY predicted, n DESC;